# Silver Line Phase 1: Transit Accessibility in 2014 vs 2021

Three maps side by side, for a large regional view and for close-ups of the Phase 1 stations, plus a **swipe map** to slide between 2014 and 2021.

**The three maps**
1. **2014**: jobs accessible within 30 min by transit. White = little or none, darker blue = more.
2. **2021**: the same measure, same colors and scale.
3. **Change, 2014 to 2021 (percent)**: **blue** = growth, **white** = no change (within +/-1%), **light red** = decline.

**Your data (upload all five zips as they are)**
- `WA_transit_accessibility_data_2014_geopackage.zip`: 2014 census-block shapes with `tot_jobs`
- `Virginia_transit_2021.zip`, `District_of_Columbia_transit_2021.zip`: 2021 block CSVs
- `Metro_Lines_Regional.zip`, `Metro_Stations_Regional_1.zip`

**Read this first**
- The comparison year is **2021** (the second year in your data).
- Only blocks with a positive value in **both** years are compared (gray = not comparable). Blocks with no resident workers are stored as 0 in 2021, so they are left out. 2021 has **no Maryland data**.
- **Gray = cannot be measured.** Blocks with no resident workers (for example the commercial core of Tysons) have no 2021 value, so they are left gray. Station percentages come from the comparable blocks around each station.
- The years differ in departure time (7:00-8:59 AM in 2014, 8:00 AM in 2021) and 2021 reflects COVID-era service, so changes are indicative, not proof of what the Silver Line caused.
- The **large view** groups blocks into half-mile hexagons (median of the blocks inside) so the pattern is readable; the **close-ups** show individual blocks. Both can be changed in the settings.

## 1. Install and import

In [ ]:
!pip -q install contextily mapclassify pyogrio

import os, glob, zipfile, base64
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import Polygon, box
import contextily as cx

plt.rcParams["figure.dpi"] = 110

## 2. Upload the five zip files
Select all five at once. Zips are extracted automatically.

In [ ]:
from google.colab import files

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    path = os.path.join(DATA_DIR, name)
    with open(path, "wb") as f:
        f.write(content)
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(os.path.join(DATA_DIR, os.path.splitext(name)[0]))

# --- Alternative: read from Google Drive instead of uploading ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/your_folder"   # unzip the files there first

found = [p for p in glob.glob(f"{DATA_DIR}/**/*", recursive=True)
         if p.lower().endswith((".shp", ".csv")) and "__MACOSX" not in p]
print("\n".join(sorted(found)))

## 3. Settings

In [ ]:
def find_file(ext, include):
    hits = [p for p in glob.glob(f"{DATA_DIR}/**/*{ext}", recursive=True)
            if include.lower() in os.path.basename(p).lower()
            and "__MACOSX" not in p and not os.path.basename(p).startswith(".")]
    assert hits, f"No {ext} file with '{include}' in its name was found. Check the upload."
    return hits[0]

SHP_2014     = find_file(".shp", "tr_2014")
LINES_SHP    = find_file(".shp", "Lines")
STATIONS_SHP = find_file(".shp", "Stations")
DC_CSV       = find_file("_block_2021.csv", "District")
VA_CSV       = find_file("_block_2021.csv", "Virginia")

THRESHOLD_SEC = 1800                    # 30 minutes
LABEL = "Jobs accessible within 30 min by transit"

# Smoothing: None = map every census block (finest detail, but speckled).
# A number (e.g. 0.5) groups blocks into hexagons of that width in miles and maps each hexagon's median.
HEX_MILES_LARGE   = 0.5                 # large view and swipe map
HEX_MILES_CLOSEUP = None                # close-ups (try 0.15 for smoother close-ups)
MIN_BLOCKS = 2                          # hexagons with fewer comparable blocks are left blank

# Silver Line
LINE_NAME_COL, LINE_NAME_VALUE = "NAME", "silver"
STATION_LINE_COL, STATION_NAME_COL = "LINE", "NAME"
PHASE1_KEYS = ["wiehle", "spring hill", "greensboro", "tysons", "mclean"]                 # opened 26 Jul 2014
PHASE2_KEYS = ["reston town", "herndon", "innovation", "dulles", "loudoun", "ashburn"]   # opened Nov 2022

# Level maps (white -> blue)
BREAKS = [0, 100, 1_000, 5_000, 25_000, 100_000, 250_000, np.inf]
LEVEL_COLORS = list(plt.get_cmap("Blues")(np.linspace(0.04, 1.0, len(BREAKS) - 1)))

# Change map: blue = growth, white = no change, light red = decline
PCT_BREAKS = [-np.inf, -25, -10, -3, -1, 1, 3, 10, 25, 50, np.inf]
PCT_LABELS = ["-25% or worse", "-25 to -10", "-10 to -3", "-3 to -1", "-1 to +1 (no change)",
              "+1 to +3", "+3 to +10", "+10 to +25", "+25 to +50", "+50% or more"]
CHG_COLORS = ["#f28e75", "#f7b3a1", "#fbd0c4", "#fde8e2", "#ffffff",
              "#d6e8f7", "#a9cfe9", "#6baed6", "#3182bd", "#08519c"]

LINE_COLOR = "#e6007e"
NOCOMP = "#bdbdbd"
BASEMAPS = [("CartoDB Positron",  cx.providers.CartoDB.Positron),      # all keyless
            ("OpenStreetMap",     cx.providers.OpenStreetMap.Mapnik),
            ("Esri WorldStreetMap", cx.providers.Esri.WorldStreetMap)]

## 4. Load both years and match them block by block

In [ ]:
acc = gpd.read_file(SHP_2014)[["geoid", "tot_jobs", "geometry"]].rename(columns={"tot_jobs": "j14"})
acc["geoid"] = acc["geoid"].astype(str)
acc.loc[acc["j14"] < 0, "j14"] = np.nan                       # -1 = no data in the 2014 file

def read_rows(path):
    parts = []
    for ch in pd.read_csv(path, dtype={"geoid": str}, chunksize=500_000):
        parts.append(ch[ch["threshold"].isin([THRESHOLD_SEC, 3600])][["geoid", "threshold", "weighted_average"]])
    return pd.concat(parts)

raw = pd.concat([read_rows(DC_CSV), read_rows(VA_CSV)])
raw["threshold"] = raw["threshold"].astype(int)
v = raw.pivot_table(index="geoid", columns="threshold", values="weighted_average", aggfunc="first")
j21 = v[THRESHOLD_SEC].where(v[3600] > 0)                      # 0 even at 60 min = no resident workers -> undefined
acc = acc.merge(j21.rename("j21").reset_index(), on="geoid", how="left")

acc["valid"] = (acc["j14"] > 0) & (acc["j21"] > 0)
acc["j14m"] = acc["j14"].where(acc["valid"])
acc["j21m"] = acc["j21"].where(acc["valid"])
acc["pct_chg"] = np.where(acc["valid"], (acc["j21"] / acc["j14"] - 1) * 100, np.nan)
acc["state"] = acc["geoid"].str[:2].map({"11": "DC", "24": "Maryland", "51": "Virginia", "54": "West Virginia"})
acc = acc.to_crs(3857)

tbl = acc.groupby("state").agg(blocks=("geoid", "size"), comparable=("valid", "sum"))
tbl["share comparable"] = (tbl["comparable"] / tbl["blocks"]).round(2)
display(tbl)

## 5. Silver Line, stations, and change near each station

In [ ]:
silver = gpd.read_file(LINES_SHP).to_crs(3857)
silver = silver[silver[LINE_NAME_COL].str.lower() == LINE_NAME_VALUE]
stn = gpd.read_file(STATIONS_SHP).to_crs(3857)
stn = stn[stn[STATION_LINE_COL].str.contains(LINE_NAME_VALUE, case=False, na=False)].copy()

P1  = "Phase 1 stations (opened Jul 2014)"
P2  = "Phase 2 stations (not open in 2014 or 2021)"
OTH = "Shared stations (also Blue/Orange)"
def classify(n):
    n = str(n).lower()
    return P1 if any(k in n for k in PHASE1_KEYS) else P2 if any(k in n for k in PHASE2_KEYS) else OTH
stn["phase"] = stn[STATION_NAME_COL].apply(classify)
assert (stn["phase"] == P1).sum() == 5, "Expected 5 Phase 1 stations; check PHASE1_KEYS"

# Median change among comparable blocks near each station (0.5 mile; widened to 1 mile if fewer than 3 blocks)
CRS_M = 32618
pts = acc[acc["valid"]].copy()
pts["geometry"] = pts.to_crs(CRS_M).geometry.representative_point()
pts = pts.set_crs(CRS_M, allow_override=True)
rows = []
for _, s in stn.to_crs(CRS_M).iterrows():
    for miles in (0.5, 1.0):
        near = pts[pts.geometry.distance(s.geometry) <= miles * 1609.34]
        if len(near) >= 3: break
    ok = len(near) >= 3
    rows.append({"station": s[STATION_NAME_COL], "phase": s["phase"], "radius_mi": miles if ok else np.nan, "blocks": len(near) if ok else 0,
                 "median_2014": near["j14"].median() if ok else np.nan, "median_2021": near["j21"].median() if ok else np.nan})
st = pd.DataFrame(rows)
st["pct_change"] = (st["median_2021"] / st["median_2014"] - 1) * 100
st_label = {r["station"]: (f"{r['pct_change']:+.0f}%" + ("*" if r["radius_mi"] == 1.0 else "")) for _, r in st.iterrows() if pd.notna(r["pct_change"])}
display(st[st["phase"] == P1].round(1))

## 6. Prepare the map layers (blocks or hexagons)

In [ ]:
def make_hexes(bounds, width_miles):
    w = width_miles * 1609.34 / np.cos(np.radians(38.9))      # Web Mercator metres at DC's latitude
    r = w / np.sqrt(3)
    minx, miny, maxx, maxy = bounds
    polys, row, y = [], 0, miny
    while y < maxy + r:
        x = minx + (w / 2 if row % 2 else 0)
        while x < maxx + w:
            polys.append(Polygon([(x + r * np.cos(np.radians(30 + 60 * k)), y + r * np.sin(np.radians(30 + 60 * k))) for k in range(6)]))
            x += w
        y += 1.5 * r; row += 1
    return gpd.GeoDataFrame({"hex_id": range(len(polys))}, geometry=polys, crs=3857)

def build_plot_df(hex_miles):
    if hex_miles:
        hx = make_hexes(acc.total_bounds, hex_miles)
        p = acc[acc["valid"]].copy(); p["geometry"] = p.geometry.representative_point()
        j = gpd.sjoin(p[["j14", "j21", "geometry"]], hx, how="inner", predicate="within")
        g = j.groupby("hex_id").agg(n=("j14", "size"), j14m=("j14", "median"), j21m=("j21", "median")).reset_index()
        g = g[g["n"] >= MIN_BLOCKS]
        g["pct_chg"] = (g["j21m"] / g["j14m"] - 1) * 100
        df = hx.merge(g, on="hex_id", how="inner")
        print(f"{len(df):,} hexagons ({hex_miles} mi) with at least {MIN_BLOCKS} comparable blocks")
    else:
        df = acc.copy()
        print(f"{len(df):,} census blocks")
    df["cls14"] = pd.cut(df["j14m"], BREAKS, right=False, labels=False)
    df["cls21"] = pd.cut(df["j21m"], BREAKS, right=False, labels=False)
    df["clsp"]  = pd.cut(df["pct_chg"], PCT_BREAKS, right=False, labels=False)
    return df[df.geometry.notna()]

df_large = build_plot_df(HEX_MILES_LARGE)
df_close = build_plot_df(HEX_MILES_CLOSEUP)

## 7. Map helpers

In [ ]:
STYLE = {
    P1:  dict(color="white",   edgecolor=LINE_COLOR, marker="o", markersize=85, linewidth=2.6, zorder=7),
    P2:  dict(color="white",   edgecolor="#333333",  marker="s", markersize=55, linewidth=1.6, zorder=6),
    OTH: dict(color="#d0d0d0", edgecolor="#333333",  marker="o", markersize=26, linewidth=0.9, zorder=6),
}
LEVEL_LABELS = [f"{lo:,.0f} to {hi:,.0f}" if np.isfinite(hi) else f"{lo:,.0f}+" for lo, hi in zip(BREAKS[:-1], BREAKS[1:])]

def add_basemap_safe(ax):
    for name, src in BASEMAPS:
        try:
            cx.add_basemap(ax, source=src, crs=3857, zorder=0, attribution_size=6)
            return name
        except Exception as e:
            print(f"Basemap '{name}' failed ({type(e).__name__}); trying next...")
    print("WARNING: no basemap could be loaded. Maps are drawn without a background.")

def draw_classes(ax, data, cls_col, colors, alpha=0.82):
    for i, c in enumerate(colors):
        sub = data[data[cls_col] == i]
        if len(sub): sub.plot(ax=ax, color=c, alpha=alpha, linewidth=0, zorder=2)
    other = data[data[cls_col].isna()]
    if len(other): other.plot(ax=ax, color=NOCOMP, alpha=0.35, linewidth=0, zorder=2)

def draw_silver(ax):
    silver.plot(ax=ax, color="white", linewidth=7, zorder=5)
    silver.plot(ax=ax, color=LINE_COLOR, linewidth=3.5, zorder=6)
    for k, kw in STYLE.items():
        sub = stn[stn["phase"] == k]
        if len(sub): sub.plot(ax=ax, **kw)

LABEL_OFFSETS = {"Wiehle-Reston East": (0, 16), "Spring Hill": (-36, -30), "Greensboro": (-34, 20),
                 "Tysons": (6, -34), "McLean": (26, 22), "East Falls Church": (-4, -30)}   # label offsets in points

def draw_labels(ax, names, with_pct, fs=8):
    lab = stn[stn[STATION_NAME_COL].isin(names)].assign(x=lambda d: d.geometry.x).sort_values("x")
    for j, (_, r) in enumerate(lab.iterrows()):
        txt = r[STATION_NAME_COL]
        if with_pct and txt in st_label: txt += f"\n{st_label[txt]}"
        off = LABEL_OFFSETS.get(r[STATION_NAME_COL], (0, 12 if j % 2 == 0 else -22))
        ax.annotate(txt, (r.geometry.x, r.geometry.y), xytext=off, textcoords="offset points",
                    ha="center", fontsize=fs, fontweight="bold" if with_pct else "normal", zorder=9, linespacing=0.95,
                    path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

def station_handles():
    h = [Line2D([0], [0], color=LINE_COLOR, lw=4, label="Silver Line")]
    for k, kw in STYLE.items():
        h.append(Line2D([0], [0], marker=kw["marker"], color="w", markerfacecolor=kw["color"],
                        markeredgecolor=kw["edgecolor"], markeredgewidth=kw["linewidth"], markersize=9, label=k))
    return h

def swatches(colors, labels):
    h = [Patch(facecolor=c, edgecolor="#999999" if mcolors.to_hex(c) == "#ffffff" else c, linewidth=0.8, label=l)
         for c, l in zip(colors, labels)]
    h.append(Patch(facecolor=NOCOMP, alpha=0.5, label="Not comparable / no data"))
    return h

def three_panel(bounds, fname, title, label_names, df, label_fs=8, width=24):
    minx, miny, maxx, maxy = bounds
    panel_w = width / 3 * 0.97
    panel_h = panel_w * (maxy - miny) / (maxx - minx)
    legend_h = 1.9
    fig_h = panel_h + legend_h + 1.3
    fig = plt.figure(figsize=(width, fig_h))
    gs = fig.add_gridspec(2, 3, height_ratios=[panel_h, legend_h], hspace=0.03, wspace=0.02,
                          top=1 - 1.0 / fig_h, bottom=0.01, left=0.005, right=0.995)
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    data = df.cx[minx:maxx, miny:maxy]
    panels = [("cls14", LEVEL_COLORS, "2014", 0.82), ("cls21", LEVEL_COLORS, "2021", 0.82),
              ("clsp", CHG_COLORS, "Change, 2014 to 2021 (percent)", 0.92)]
    for i, (ax, (col, colors, ttl, a)) in enumerate(zip(axes, panels)):
        ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
        draw_classes(ax, data, col, colors, a)
        draw_silver(ax)
        add_basemap_safe(ax)
        draw_labels(ax, label_names, with_pct=(i == 2), fs=label_fs)
        ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy); ax.set_aspect("auto"); ax.set_axis_off()
        ax.set_title(ttl, fontsize=15, fontweight="bold")
    la = fig.add_subplot(gs[1, 0:2]); la.axis("off")
    l1 = la.legend(handles=swatches(LEVEL_COLORS, LEVEL_LABELS), loc="upper center", ncol=4, frameon=False, fontsize=9,
                   title=f"{LABEL} (white = little or none, dark blue = most)", title_fontsize=10)
    la.add_artist(l1)
    la.legend(handles=station_handles(), loc="lower center", ncol=4, frameon=False, fontsize=9)
    lb = fig.add_subplot(gs[1, 2]); lb.axis("off")
    lb.legend(handles=swatches(CHG_COLORS, PCT_LABELS), loc="upper center", ncol=3, frameon=False, fontsize=9,
              title="Percent change (blue = growth, light red = decline)", title_fontsize=10)
    if any(st_label.get(n, "").endswith("*") for n in label_names):
        lb.text(0.5, -0.02, "* station change measured within 1 mile (too few blocks within 0.5 mile)", ha="center", va="top",
                fontsize=8, color="#555555", transform=lb.transAxes)
    fig.suptitle(title, fontsize=18, fontweight="bold", y=1 - 0.1 / fig_h)
    fig.savefig(fname, dpi=170, bbox_inches="tight"); plt.show()

def pad(b, f=0.04):
    minx, miny, maxx, maxy = b
    return (minx - (maxx - minx) * f, miny - (maxy - miny) * f, maxx + (maxx - minx) * f, maxy + (maxy - miny) * f)

p1_names = list(stn[stn["phase"] == P1][STATION_NAME_COL])
sb = silver.total_bounds
cxm, cym = (sb[0] + sb[2]) / 2, (sb[1] + sb[3]) / 2
wv = (sb[2] - sb[0]) * 1.08
large_view = (cxm - wv / 2, cym - wv / 1.45 / 2, cxm + wv / 2, cym + wv / 1.45 / 2)     # whole corridor, ~1.45:1 panels
ef = stn[stn[STATION_NAME_COL].str.contains("East Falls Church", case=False)]
phase1_view = pad(pd.concat([stn[stn["phase"] == P1], ef]).total_bounds, 0.22)
tysons_view = pad(stn[stn[STATION_NAME_COL].isin(["Spring Hill", "Greensboro", "Tysons", "McLean"])].total_bounds, 0.9)

## 8. Large view: 2014, 2021 and change

In [ ]:
three_panel(large_view, "map_A_large_view_2014_2021_change.png",
            "Jobs accessible within 30 min by transit: 2014, 2021 and change", ["Wiehle-Reston East", "Tysons"],
            df_large, label_fs=8.5)

## 9. Close-up: the Phase 1 stations (Wiehle-Reston East to East Falls Church)

In [ ]:
three_panel(phase1_view, "map_B_phase1_closeup.png",
            "Phase 1 stations: jobs accessible within 30 min by transit, 2014, 2021 and change",
            p1_names + ["East Falls Church"], df_close, label_fs=8.5)

## 10. Closer close-up: Spring Hill, Greensboro, Tysons and McLean

In [ ]:
three_panel(tysons_view, "map_C_tysons_closeup.png",
            "Tysons area: jobs accessible within 30 min by transit, 2014, 2021 and change",
            ["Spring Hill", "Greensboro", "Tysons", "McLean"], df_close, label_fs=10)

## 11. Swipe map: 2014 vs 2021 (large view)
Drag the slider (or click the map's divider area) to reveal 2021 on the right and 2014 on the left. The change map is available as an extra layer in the layer control. The swipe uses pictures of the maps, so there are no hover tooltips on individual blocks.

In [ ]:
import folium
from folium.raster_layers import ImageOverlay
from branca.element import Element

def render_overlay(col, colors, fname, alpha, width_px=2800):
    minx, miny, maxx, maxy = large_view
    h_px = int(width_px * (maxy - miny) / (maxx - minx)); dpi = 100
    fig = plt.figure(figsize=(width_px / dpi, h_px / dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_axis_off(); ax.set_aspect("auto")
    ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
    draw_classes(ax, df_large.cx[minx:maxx, miny:maxy], col, colors, alpha)
    fig.savefig(fname, dpi=dpi, transparent=True); plt.close(fig)
    return "data:image/png;base64," + base64.b64encode(open(fname, "rb").read()).decode()

img14  = render_overlay("cls14", LEVEL_COLORS, "swipe_2014.png", 0.8)
img21  = render_overlay("cls21", LEVEL_COLORS, "swipe_2021.png", 0.8)
imgchg = render_overlay("clsp",  CHG_COLORS,   "swipe_change.png", 0.9)

w_, s_, e_, n_ = gpd.GeoSeries([box(*large_view)], crs=3857).to_crs(4326).total_bounds
bnds = [[s_, w_], [n_, e_]]
m = folium.Map(location=[(s_ + n_) / 2, (w_ + e_) / 2], zoom_start=10, tiles="CartoDB positron", control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap basemap", show=False).add_to(m)
ov14 = ImageOverlay(img14, bounds=bnds, opacity=1, control=False, zindex=10).add_to(m)
ov21 = ImageOverlay(img21, bounds=bnds, opacity=1, control=False, zindex=11).add_to(m)
ImageOverlay(imgchg, bounds=bnds, opacity=1, name="Percent change 2014 to 2021 (covers the swipe)", show=False, zindex=12).add_to(m)

folium.GeoJson(silver.to_crs(4326).to_json(), name="Silver Line", style_function=lambda f: {"color": LINE_COLOR, "weight": 5}).add_to(m)
for _, r in stn.to_crs(4326).iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=7 if r["phase"] != OTH else 4,
                        color=LINE_COLOR if r["phase"] == P1 else "#333333", weight=3, fill=True, fill_color="white",
                        fill_opacity=1, tooltip=f"{r[STATION_NAME_COL]} | {r['phase']}").add_to(m)
folium.LayerControl(collapsed=True).add_to(m)

ui = """
<style>
 #swipe-line {position:absolute; top:0; bottom:0; width:4px; margin-left:-2px; background:#fff; box-shadow:0 0 4px #000; z-index:900; pointer-events:none;}
 #swipe-wrap {position:absolute; left:50%; bottom:28px; transform:translateX(-50%); z-index:1000; background:rgba(255,255,255,.95);
              padding:8px 14px; border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.4); font:13px sans-serif; text-align:center; width:min(520px,80%);}
 #swipe-wrap input {width:100%;}
 #swipe-legend {position:absolute; left:12px; bottom:28px; z-index:1000; background:rgba(255,255,255,.95); padding:8px 10px; border-radius:6px;
                box-shadow:0 1px 6px rgba(0,0,0,.4); font:12px sans-serif; line-height:1.5;}
 #swipe-legend i {display:inline-block; width:18px; height:11px; margin-right:6px; border:1px solid #bbb; vertical-align:middle;}
 .swipe-tag {position:absolute; top:12px; z-index:1000; background:rgba(255,255,255,.95); padding:4px 10px; border-radius:4px;
             font:bold 15px sans-serif; box-shadow:0 1px 4px rgba(0,0,0,.4); pointer-events:none;}
</style>
<div id="swipe-line"></div><div class="swipe-tag" style="left:56px">2014</div><div class="swipe-tag" style="right:56px">2021</div>
<div id="swipe-legend"><b>Jobs accessible within 30 min by transit</b><br>__LEGEND__<br><span style="color:#777">white = little or none</span></div>
<div id="swipe-wrap">Jobs accessible within 30 min by transit<br><input id="swipe-range" type="range" min="0" max="100" value="50"><br>
<span style="color:#555">&larr; 2014 | 2021 &rarr; (drag the slider)</span></div>
"""
legend_html = "<br>".join(f'<i style="background:{mcolors.to_hex(c)}"></i>{l}' for c, l in zip(LEVEL_COLORS, LEVEL_LABELS))
ui = ui.replace("__LEGEND__", legend_html)
js = """
window.addEventListener('load', function(){
  var map = %(map)s, right = %(right)s;
  var host = map.getContainer();
  host.insertAdjacentHTML('beforeend', %(ui)s);
  var slider = document.getElementById('swipe-range'), line = document.getElementById('swipe-line');
  function update(){
    var el = right.getElement(); if(!el) return;
    var frac = slider.value / 100, mrect = host.getBoundingClientRect();
    var xs = mrect.width * frac;                                  // divider position inside the map box
    line.style.left = xs + 'px';
    var r = el.getBoundingClientRect();
    var scale = r.width > 0 ? el.offsetWidth / r.width : 1;       // undo any zoom-animation scaling
    var local = (mrect.left + xs - r.left) * scale;
    el.style.clipPath = 'inset(0px 0px 0px ' + Math.max(0, local) + 'px)';
  }
  slider.addEventListener('input', update);
  map.on('move zoom zoomend moveend viewreset resize layeradd', update);
  setTimeout(update, 50); setTimeout(update, 500);
});
""" % {"map": m.get_name(), "right": ov21.get_name(), "ui": __import__("json").dumps(ui)}
m.get_root().script.add_child(Element(js))
m.save("swipe_map_2014_vs_2021.html")
print("Saved swipe_map_2014_vs_2021.html")
m

## 12. Station numbers
Median jobs accessible near each station (comparable blocks only).

In [ ]:
display(st.sort_values("pct_change", ascending=False).round(1).reset_index(drop=True))
st.to_csv("station_change_2014_2021.csv", index=False)

## 13. Download everything

In [ ]:
for f in sorted(glob.glob("map_*.png") + glob.glob("swipe_map_*.html") + glob.glob("station_change_*.csv")):
    files.download(f)